In [1]:
import pandas as pd
import sqlite3

In [2]:
from pathlib import Path

# Path.home() automatically finds your Windows User directory (e.g., C:\Users\YourName)
file_path = Path.home() / "Downloads" / "YULA Financial Management - 2025-2026 - BankAccount-cleaned.csv"


df = pd.read_csv(file_path)
df.head()

,Date,Transaction Type,Check Number,Description,Amount,Daily Posted Balance,Fiscal Year,Category,Notes,More Notes,...,Fall.1,Spring.1,Unnamed: 15,Running Calculated Balance,Delta,Fall.2,Winter.1,Spring.2,Fall.3,Spring.3
0,2017-09-01,credit,NaN,Carry Forward,"$11,143.93","$11,143.93",FY2017,carry forward,Export from NPTreasurer,NaN,...,NaN,NaN,NEEDS ALLOCAITON,"$11,143.93",$0.00,NaN,NaN,NaN,NaN,NaN
1,2017-09-01,credit,NaN,170901P2 Square Inc 2963 David T,$28.83,"$11,172.76",FY2018,Merchandise Sales,Export from NPTreasurer,NaN,...,NaN,NaN,NEEDS ALLOCAITON,"$11,172.76",$0.00,NaN,NaN,NaN,NaN,NaN
2,2017-09-12,debit,NaN,PAYMENT TO CREDIT CARD *********,-$0.43,"$11,172.33",FY2018,Admin/Supplies,Export from NPTreasurer,NaN,...,NaN,NaN,NEEDS ALLOCAITON,"$11,172.33",$0.00,NaN,NaN,NaN,NaN,NaN
3,2017-09-12,credit,NaN,COUNTER DEPOSIT,$607.00,"$11,779.33",FY2018,Other Income,Export from NPTreasurer,NaN,...,NaN,NaN,NEEDS ALLOCAITON,"$11,779.33",$0.00,NaN,NaN,NaN,NaN,NaN
4,2017-09-13,debit,NaN,RETURN DEPOSIT ITEM,-$40.00,"$11,739.33",FY2018,Admin/Supplies,Export from NPTreasurer,NaN,...,NaN,NaN,NEEDS ALLOCAITON,"$11,739.33",$0.00,NaN,NaN,NaN,NaN,NaN


In [3]:
conn = sqlite3.connect("data/ledger.db")
# Fetch {name: id} pairs from categories table
cat_map = pd.read_sql("SELECT name, id FROM categories", conn).set_index('name')['id'].to_dict()
cat_map

{'Admin/Supplies': 15,
 'Better Sports Club': 20,
 'Bids': 10,
 'Buses': 11,
 'Coaches': 12,
 'Concussion Testing': 21,
 'Diversity, Equity, Inclusion': 7,
 'Donation': 1,
 'Fields': 9,
 'Fundraising': 2,
 'Hotels': 13,
 'Insurance': 14,
 'Merchandise Purchases': 16,
 'Merchandise Sales': 6,
 'Other Expenses': 23,
 'Other Income': 8,
 'Outreach': 24,
 'Registration': 3,
 'Richmond Rendezvous': 22,
 'Richmond Rendezvous Bids': 5,
 'Team Stipends': 17,
 'Uniforms': 18,
 'YULA Invite': 19,
 'YULA Invite Bids': 4}

In [4]:
cat_map["Concussion"] = 21
cat_map["Richmond Rendevous"] = 22
cat_map["Richmond Rendevous Bids"] = 5
cat_map["carry forward"] = 8
cat_map["Other"] = 23
cat_map["YCC"] = 23
cat_map["Yula Invite"] = 19
cat_map["Yula Invite Bids"] = 4
#cat_map["Outreach"] = 
cat_map

{'Admin/Supplies': 15,
 'Better Sports Club': 20,
 'Bids': 10,
 'Buses': 11,
 'Coaches': 12,
 'Concussion Testing': 21,
 'Diversity, Equity, Inclusion': 7,
 'Donation': 1,
 'Fields': 9,
 'Fundraising': 2,
 'Hotels': 13,
 'Insurance': 14,
 'Merchandise Purchases': 16,
 'Merchandise Sales': 6,
 'Other Expenses': 23,
 'Other Income': 8,
 'Outreach': 24,
 'Registration': 3,
 'Richmond Rendezvous': 22,
 'Richmond Rendezvous Bids': 5,
 'Team Stipends': 17,
 'Uniforms': 18,
 'YULA Invite': 19,
 'YULA Invite Bids': 4,
 'Concussion': 21,
 'Richmond Rendevous': 22,
 'Richmond Rendevous Bids': 5,
 'carry forward': 8,
 'Other': 23,
 'YCC': 23,
 'Yula Invite': 19,
 'Yula Invite Bids': 4}

In [5]:
def clean_currency(x):
    if isinstance(x, str):
        x = x.replace('$', '').replace(',', '')
        if '(' in x or '-' in x:
            return -float(x.replace('(', '').replace(')', '').replace('-', ''))
        return float(x)
    return x

df['amount'] = df['Amount'].apply(clean_currency)
df['daily posted balance'] = df['Daily Posted Balance'].apply(clean_currency)
df['running balance'] = df['Running Calculated Balance'].apply(clean_currency)




In [6]:
# Map CSV 'Category' to category_id
# If not found, it defaults to None (NULL in SQL)
df['category_id'] = df['Category'].map(cat_map)

In [ ]:
df.head()

In [ ]:

# Group by columns and count rows in each group
#cat_column = df.groupby(['Category', 'category_id']).size().reset_index(name='count')
# Set dropna=False to include rows where column_b (or column_a) is null
cat_column = df.groupby(['Category', 'category_id'], dropna=False).size().reset_index(name='count')

cat_column


In [ ]:
cat_id = df[['Category', 'category_id']].value_counts(dropna=False).reset_index(name='count')
cat_id

In [ ]:

# Group by columns and count rows in each group
cat_id = df.groupby(['category_id']).size().reset_index(name='count')
cat_id


In [7]:

# Fetch {name: id} pairs from categories table
fy_map = pd.read_sql("SELECT name, fiscal_year_id FROM fy", conn).set_index('name')['fiscal_year_id'].to_dict()
fy_map

{'FY2017': 1,
 'FY2018': 2,
 'FY2019': 3,
 'FY2020': 4,
 'FY2021': 5,
 'FY2022': 6,
 'FY2023': 7,
 'FY2024': 8,
 'FY2025': 9,
 'FY2026': 10}

In [29]:
# Map CSV 'Fiscal Year' to fiscal_year_id
# If not found, it defaults to None (NULL in SQL)
df['fiscal_year_id'] = df['Fiscal Year'].map(fy_map)
df.head()

,Date,Transaction Type,Check Number,Description,Amount,Daily Posted Balance,Fiscal Year,Category,Notes,More Notes,Fall,Winter,Spring,Fall.1,Spring.1,Unnamed: 15,Running Calculated Balance,Delta,Fall.2,Winter.1,Spring.2,Fall.3,Spring.3,amount,daily posted balance,running balance,category_id,allocation_method_id,fiscal_year_id
0,2017-09-01,credit,NaN,Carry Forward,"$11,143.93","$11,143.93",FY2017,carry forward,Export from NPTreasurer,NaN,NaN,NaN,NaN,NaN,NaN,NEEDS ALLOCAITON,"$11,143.93",$0.00,NaN,NaN,NaN,NaN,NaN,11143.93,11143.93,11143.93,8,None,1
1,2017-09-01,credit,NaN,170901P2 Square Inc 2963 David T,$28.83,"$11,172.76",FY2018,Merchandise Sales,Export from NPTreasurer,NaN,NaN,NaN,NaN,NaN,NaN,NEEDS ALLOCAITON,"$11,172.76",$0.00,NaN,NaN,NaN,NaN,NaN,28.83,11172.76,11172.76,6,None,2
2,2017-09-12,debit,NaN,PAYMENT TO CREDIT CARD *********,-$0.43,"$11,172.33",FY2018,Admin/Supplies,Export from NPTreasurer,NaN,NaN,NaN,NaN,NaN,NaN,NEEDS ALLOCAITON,"$11,172.33",$0.00,NaN,NaN,NaN,NaN,NaN,-0.43,11172.33,11172.33,15,None,2
3,2017-09-12,credit,NaN,COUNTER DEPOSIT,$607.00,"$11,779.33",FY2018,Other Income,Export from NPTreasurer,NaN,NaN,NaN,NaN,NaN,NaN,NEEDS ALLOCAITON,"$11,779.33",$0.00,NaN,NaN,NaN,NaN,NaN,607.00,11779.33,11779.33,8,None,2
4,2017-09-13,debit,NaN,RETURN DEPOSIT ITEM,-$40.00,"$11,739.33",FY2018,Admin/Supplies,Export from NPTreasurer,NaN,NaN,NaN,NaN,NaN,NaN,NEEDS ALLOCAITON,"$11,739.33",$0.00,NaN,NaN,NaN,NaN,NaN,-40.00,11739.33,11739.33,15,None,2


In [8]:
# Fetch allocation methods
allocations = pd.read_sql("SELECT * FROM allocation_methods", conn)
allocations

,id,name,description
0,1,High School Fall,Direct entirely High School Fall
1,2,High School Winter,Direct entirely High School Winter
2,3,High School Spring,Direct entirely High School Spring
3,4,Middle School Fall,Direct entirely Middle School Fall
4,5,Middle School Spring,Direct entirely Middle School Spring
5,6,2026 Player Distribution,Split based on the distribution of players in ...
6,7,2026 Registration Distribution,Split based on the distribution of registratio...
7,8,2025 Player Distribution,Split based on the distribution of players in ...
8,9,2025 Registration Distribution,Split based on the distribution of registratio...
9,10,High School Season Split,For proportional between the High School fall ...


In [24]:
# let's see what combinations we have in the df:
# Filtering (WHERE) -> Grouping -> Selecting Column -> Aggregating
filter_condition = df['Fiscal Year'].isin(['FY2026'])
alloc = df[df['Fiscal Year'].isin(['FY2026','FY2025'])].groupby(['Fiscal Year','Fall','Winter','Spring','Fall.1','Spring.1'], dropna=False)['Fiscal Year'].count()
alloc

Fiscal Year  Fall    Winter  Spring        Fall.1  Spring.1    
FY2025       100%    NaN     NaN           NaN     NaN             44
             13%     4%      56%           9%      18%             29
             24%     4%      48%           8%      16%             13
             24.05%  12.37%  32.65%        10.82%  20.10%           1
             30%     NaN     70%           NaN     NaN              6
                             NaN           70%     NaN              1
             35%     NaN     65%           NaN     NaN             13
             69%     NaN     NaN           31%     NaN              1
             70%     NaN     NaN           30%     NaN             55
             NaN     100%    NaN           NaN     NaN             10
                     NaN     0.6086956522  NaN     0.3913043478     1
                             100%          NaN     NaN             97
                             85.80%        NaN     14.20%          13
                          

In [25]:
# Map for obvious groups that can be mapped:
def classify_allocation(row):
    if row['Fall'] == "100%":
        return "1" # High School Fall
    elif row['Winter'] == "100%":
        return "2" # High School Winter
    elif row['Spring'] == "100%":
        return "3" # High School Spring
    elif row['Fall'] == "35%" and row['Spring'] == "65%":
        return "10" # High School Split
    elif row['Fall.1'] == "100%":
        return "4" # Middle School Fall
    elif row['Spring.1'] == "100%":
        return "5" # Middle School Spring
    elif row['Fall'] == "13%" and row['Fiscal Year'] == "FY2025":
        return "9" # Registration Distribution
    elif row['Fall'] == "24%" and row['Fiscal Year'] == "FY2025":
        return "9" # Registration Distribution
    elif row['Fall'] == "13%" and row['Fiscal Year'] == "FY2026":
        return "7" # Registration Distribution
    elif row['Fall'] == "24.05%" and row['Fiscal Year'] == "FY2025":
        return "8" # Player Distribution
    elif row['Fall'] == "24.05%" and row['Fiscal Year'] == "FY2026":
        return "6" # Player Distribution
    else:
        return None

df['allocation_method_id'] = df.apply(classify_allocation, axis=1)

In [26]:
pd.set_option('display.max_columns', None)
print(df.tail())

            Date Transaction Type Check Number  \
2630  2026-05-11            Debit          NaN   
2631  2026-05-11            Debit          NaN   
2632     pending              NaN          NaN   
2633     pending              NaN          NaN   
2634     pending              NaN          NaN   

                                       Description      Amount  \
2630  PAYMENT VENMO *********1318 INTERNET PAYMENT     -$50.00   
2631  PAYMENT VENMO *********9961 INTERNET PAYMENT     -$27.00   
2632                                           NaN  -$1,500.00   
2633                                           NaN    -$381.75   
2634                                           NaN  -$4,545.00   

     Daily Posted Balance Fiscal Year       Category                 Notes  \
2630          $117,804.85      FY2026  Team Stipends          Spring Fling   
2631          $117,804.85      FY2026        Coaches            USAU Certs   
2632          $116,304.85      FY2026    YULA Invite          TD Hon

In [27]:
# let's see what combinations we have in the df:
# Filtering (WHERE) -> Grouping -> Selecting Column -> Aggregating
filter_condition = df['Fiscal Year'].isin(['FY2026'])
alloc = df[df['Fiscal Year'].isin(['FY2026','FY2025'])].groupby(['allocation_method_id','Fiscal Year','Fall','Winter','Spring','Fall.1','Spring.1'], dropna=False)['Fiscal Year'].count()
alloc

allocation_method_id  Fiscal Year  Fall    Winter  Spring        Fall.1  Spring.1    
1                     FY2025       100%    NaN     NaN           NaN     NaN             44
                      FY2026       100%    NaN     NaN           NaN     NaN             36
10                    FY2025       35%     NaN     65%           NaN     NaN             13
                      FY2026       35%     NaN     65%           NaN     NaN             15
2                     FY2025       NaN     100%    NaN           NaN     NaN             10
                      FY2026       NaN     100%    NaN           NaN     NaN             18
3                     FY2025       NaN     NaN     100%          NaN     NaN             97
                      FY2026       NaN     NaN     100%          NaN     NaN             87
4                     FY2025       NaN     NaN     NaN           100%    NaN              2
                      FY2026       NaN     NaN     NaN           100%    NaN          

In [30]:
# 4. Prepare for Ledger
ledger_df = pd.DataFrame({
    'transaction_date': df['Date'],
    'description': df['Description'],
    'amount': df['amount'],
    'transaction_type': df['Transaction Type'],
    'check_number': df['Check Number'],
    'daily_posted_balance': df['daily posted balance'],
    'running_balance': df['running balance'],
    'category_id': df['category_id'],
    'fiscal_year_id': df['fiscal_year_id'],
    'allocation_method_id': df['allocation_method_id'],
    'source_indicator': 'Import',
    'is_deleted': 0,
    'notes': df['Notes'],
    'more_notes': df['More Notes']
    # Add other fields here as needed
})

In [31]:
# Keeps rows where 'column_name' is NOT null
df_filtered = ledger_df[ledger_df['amount'].notnull()]
df_filtered.head()

,transaction_date,description,amount,transaction_type,check_number,daily_posted_balance,running_balance,category_id,fiscal_year_id,allocation_method_id,source_indicator,is_deleted,notes,more_notes
0,2017-09-01,Carry Forward,11143.93,credit,NaN,11143.93,11143.93,8,1,None,Import,0,Export from NPTreasurer,NaN
1,2017-09-01,170901P2 Square Inc 2963 David T,28.83,credit,NaN,11172.76,11172.76,6,2,None,Import,0,Export from NPTreasurer,NaN
2,2017-09-12,PAYMENT TO CREDIT CARD *********,-0.43,debit,NaN,11172.33,11172.33,15,2,None,Import,0,Export from NPTreasurer,NaN
3,2017-09-12,COUNTER DEPOSIT,607.00,credit,NaN,11779.33,11779.33,8,2,None,Import,0,Export from NPTreasurer,NaN
4,2017-09-13,RETURN DEPOSIT ITEM,-40.00,debit,NaN,11739.33,11739.33,15,2,None,Import,0,Export from NPTreasurer,NaN


In [32]:
cursor = conn.cursor()
cursor.execute("delete from ledger")

In [33]:
# 5. Insert
df_filtered.to_sql('ledger', conn, if_exists='append', index=False)

# when replacing, it gets rid of all the existing columns and only has these ones...
# df_filtered.to_sql('ledger', conn, if_exists='replace', index=False)


In [ ]:
conn.close()
print("Import complete with category mapping applied.")